# Quick demo

Notebook takes 5.14 seconds to run on MacBook Air M1 2020

In [ ]:
# Import relevant packages and set seed
import os
import time
import numpy as np
import ewstools
from ewstools.models import simulate_ricker
from tensorflow.keras.models import load_model

np.random.seed(0)
t_start = time.time()

In [ ]:
# Simulate the Ricker model and plot trajectory
series = simulate_ricker(tmax=500, F=[0,2.7])
series.plot();

In [ ]:
# Initialize time series object
ts = ewstools.TimeSeries(data=series, transition=440)

In [ ]:
# Detrend and compute EWS
ts.detrend(method='Lowess', span=0.2)
ts.compute_var(rolling_window=0.5)
ts.compute_auto(lag=1, rolling_window=0.5)
ts.compute_auto(lag=2, rolling_window=0.5)
ts.compute_ktau()

In [ ]:
# Load deep learning classifiers (just the first two to run faster)
list_classifiers = []
path_to_classifiers = 'saved_classifiers/bury_pnas_21/len500/'
classifier_names = sorted([s for s in os.listdir(path_to_classifiers) if s[-6:]==".keras"])[:2]
for classifier_name in classifier_names:
    classifier = load_model(path_to_classifiers+classifier_name)
    list_classifiers.append(classifier)
    print(f"Loaded classifier {classifier_name}")

In [ ]:
# Get classifier predictions
for idx, classifier in enumerate(list_classifiers):
    ts.apply_classifier_inc(classifier, inc=10, verbose=0, name=str(idx))
    print('Predictions complete for classifier {}'.format(idx))

In [ ]:
bif_labels = {0:'Fold', 1:'Hopf', 2:'Transcritical', 3:'Null'}
ts.dl_preds = ts.dl_preds.rename(columns = bif_labels)

In [ ]:
# Visualize
ts.make_plotly(ens_avg=True)

In [ ]:
t_end = time.time()
print(f"Elapsed time: {t_end - t_start:.4f} seconds")